# 4. Custom GBM vs Libraries (CatBoost, LightGBM, XGBoost)

В этой части сравним готовые реализации бустинга с кастомным бустингом, вызвав у него необходимые имплементации

In [17]:
# нужные импорты
import numpy as np
import pandas as pd

import seaborn as sns
import matplotlib.pyplot as plt

import warnings
warnings.filterwarnings('ignore')
sns.set(style='darkgrid')

from sklearn.metrics import roc_auc_score

In [18]:
# для начала подгрузим данные 
columns = ['age', 'workclass', 'fnlwgt', 'education', 'education-num',
           'marital-status', 'occupation', 'relationship', 'race', 'sex',
           'capital-gain', 'capital-loss', 'hours-per-week', 'native-country', 'income']


train = pd.read_csv('data/adult.data', names=columns, skipinitialspace=True, na_values=['?'])
test = pd.read_csv('data/adult.test', names=columns, skipinitialspace=True, na_values=['?'], skiprows=1)
test['income'] = test['income'].str.rstrip('.') # тк в тесте точка на конце

In [19]:
# и заполним пропуски как мы решили это делать в EDA
train.fillna('Unknown', inplace=True)
test.fillna('Unknown', inplace=True)

In [20]:
# выделим целевую переменную
y_train = train['income'].apply(lambda x: x=='>50K').astype(int)
X_train = train.drop(columns='income', axis=1)

# отделим валидационную выборку
from sklearn.model_selection import train_test_split

X_train, X_val, y_train, y_val = train_test_split(X_train, y_train, test_size=0.2, stratify=y_train, random_state=42)

y_test = test['income'].apply(lambda x: x=='>50K').astype(int)
X_test = test.drop(columns='income', axis=1)

In [21]:
# для начала нужно предобработать признаки: закодировать категориальные признаки и стандартизовать числовые
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import StandardScaler

# взяли с EDA
cat_features = train.select_dtypes(include=['object', 'category']).columns.tolist()
num_features = train.select_dtypes(include=['int64', 'float64']).columns.tolist()

# Исключаем целевую переменную
if 'income' in cat_features:
    cat_features.remove('income')

column_transformer = ColumnTransformer([
    ('ohe', OneHotEncoder(handle_unknown='ignore'), cat_features),
    ('scaling', StandardScaler(), num_features)
])

X_train_scaled = column_transformer.fit_transform(X_train)
X_val_scaled = column_transformer.transform(X_val)
X_test_scaled = column_transformer.transform(X_test)

Для начала посмотрим на кастомный бустинг с добавленными параметрами, который мы получили ранее 

In [22]:
%load_ext autoreload

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [23]:
%autoreload 2

from boosting import Boosting

In [24]:
cat_features_idx = [X_train.columns.get_loc(col) for col in cat_features]

X_train_custom = X_train.to_numpy()
X_val_custom = X_val.to_numpy()
y_train_np = y_train.to_numpy()
y_val_np = y_val.to_numpy()

y_train_custom = np.where(y_train_np == 1, 1, -1)
y_val_custom = np.where(y_val_np == 1, 1, -1)

In [25]:
# наилучшая вариация кастомного бустинга
boosting = Boosting(
    random_state=42,
    verbose=False,

    n_estimators=500,
    learning_rate=0.1,
    base_model_params={"max_depth": 5, "min_samples_leaf": 20},

    cat_features=cat_features_idx,

    bootstrap_type="Bernoulli",
    subsample=0.75,
    rsm=0.5,

    early_stopping_rounds=15,
    eval_metric="val_roc_auc",
)

boosting.fit(X_train_custom, y_train_custom, eval_set=(X_val_custom, y_val), use_best_model=True)

In [26]:
# проверим на тесте
y_test = test['income'].apply(lambda x: x=='>50K').astype(int)
X_test = test.drop(columns='income', axis=1)

X_test = X_test.to_numpy()
y_test_np = y_test.to_numpy()

y_test = np.where(y_test_np == 1, 1, -1)

print(f"Test ROC-AUC: {boosting.score(X_test, y_test):.4f}")

Test ROC-AUC: 0.9254


Теперь посмотрим на готовые реализации бустинга. Возьмем три библиотеки: LightGBM, CatBoost, XGBoost

#### 1. LightGBM

In [27]:
pip install lightgbm

Note: you may need to restart the kernel to use updated packages.


In [28]:
import lightgbm as lgb

model = lgb.LGBMClassifier(n_estimators=500, learning_rate=0.1, colsample_bytree=0.5, random_state=42, subsample=0.75)
model.fit(
    X_train_scaled,
    y_train,
    eval_set=[(X_val_scaled, y_val)],
    eval_metric="auc",
    callbacks=[lgb.early_stopping(15)]
)

train_pred = model.predict_proba(X_train_scaled)[:, 1] 
val_pred = model.predict_proba(X_val_scaled)[:, 1] 
preds = model.predict_proba(X_test_scaled)[:, 1] 

print(f'Train ROC-AUC XGBoost {roc_auc_score(y_train, train_pred):.4f}') 
print(f'Val ROC-AUC XGBoost {roc_auc_score(y_val, val_pred):.4f}') 
print(f'Test ROC-AUC XGBoost {roc_auc_score(y_test, preds):.4f}') 

[LightGBM] [Info] Number of positive: 6273, number of negative: 19775
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.002520 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 767
[LightGBM] [Info] Number of data points in the train set: 26048, number of used features: 93
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.240825 -> initscore=-1.148164
[LightGBM] [Info] Start training from score -1.148164
Training until validation scores don't improve for 15 rounds
Early stopping, best iteration is:
[77]	valid_0's auc: 0.931401	valid_0's binary_logloss: 0.273043
Train ROC-AUC XGBoost 0.9420
Val ROC-AUC XGBoost 0.9314
Test ROC-AUC XGBoost 0.9282


#### 2. CatBoost

In [29]:
pip install catboost

Note: you may need to restart the kernel to use updated packages.


In [36]:
from catboost import CatBoostClassifier

model = CatBoostClassifier(
    learning_rate=0.1,
    random_state=42,
    early_stopping_rounds=15,
    rsm=0.5,
    n_estimators=500,
    max_depth=5,
    min_data_in_leaf=20,
    bootstrap_type='Bernoulli',
    subsample=0.75,
    loss_function='Logloss'
)
model.fit(X_train, y_train, eval_set=(X_val, y_val), cat_features=cat_features, verbose=False)

train_pred = model.predict_proba(X_train)[:, 1] 
val_pred = model.predict_proba(X_val)[:, 1] 
preds = model.predict_proba(X_test)[:, 1] 

print(f'Train ROC-AUC CatBoost {roc_auc_score(y_train, train_pred):.4f}') 
print(f'Val ROC-AUC CatBoost {roc_auc_score(y_val, val_pred):.4f}') 
print(f'Test ROC-AUC CatBoost {roc_auc_score(y_test, preds):.4f}') 

Train ROC-AUC CatBoost 0.9383
Val ROC-AUC CatBoost 0.9310
Test ROC-AUC CatBoost 0.9259


#### 3. XGBoost

In [32]:
pip install xgboost

Note: you may need to restart the kernel to use updated packages.


In [34]:
from xgboost import XGBClassifier 

model=XGBClassifier(
    booster="gbtree",
    n_estimators=500,
    learning_rate=0.1,
    max_depth=5,
    colsample_bytree=0.5,
    eval_metric="auc",
    random_state=42,
    subsample=0.75
)
model.fit(X_train_scaled, y_train, eval_set=[(X_val_scaled, y_val)], verbose=False) 

train_pred = model.predict_proba(X_train_scaled)[:, 1] 
val_pred = model.predict_proba(X_val_scaled)[:, 1] 
preds = model.predict_proba(X_test_scaled)[:, 1] 

print(f'Train ROC-AUC XGBoost {roc_auc_score(y_train, train_pred):.4f}') 
print(f'Val ROC-AUC XGBoost {roc_auc_score(y_val, val_pred):.4f}') 
print(f'Test ROC-AUC XGBoost {roc_auc_score(y_test, preds):.4f}') 

Train ROC-AUC XGBoost 0.9597
Val ROC-AUC XGBoost 0.9304
Test ROC-AUC XGBoost 0.9258


Все модели (кастомный бустинг, XGBoost, CatBoost, LightGBM) с одинаковыми гиперпараметрами дают примерно одинаковый ROC-AUC на тесте для датасета Adult census income. Из этого можно сделать следующие выводы:
- в датасете есть явная и понятная зависимость между признаками, которую улавливала даже простая логистическая регрессия 
- из EDA выяснили, что в данных нет сложной зависимости
- дополнительные имплементации смогли только немного улучшить качество модели

Кастомный бустинг не уступает готовым библиотечным реализациям по качеству предсказания, следовательно, можно сделать вывод о том, что все имплементации были реализованы корректно